In [ ]:
import re
import hashlib
import random
import pandas as pd
import os
import itertools
#Loading Documents
def load_documents_from_folder(folder_path):
    docs={}
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename),'r',encoding='utf-8') as f:
                docs[filename]=f.read()
    return docs

In [ ]:
#Printing Loaded Documents
folder_path="/content/drive/MyDrive/IR"
docs=load_documents_from_folder(folder_path)
print(f"Loaded {len(docs)} documents: {list(docs.keys())}")

Loaded 3 documents: ['doc3.txt', 'doc1.txt', 'doc2.txt']


In [ ]:
#Shingles
def get_shingles(text,k):
    text=re.sub(r'[^\w\s]','',text.lower())
    words=text.split()
    return set([' '.join(words[i:i+k]) for i in range(len(words)-k+1)])
shingles={doc: get_shingles(content, k=1) for doc, content in docs.items()}
print("\nShingles created for all documents.")
print(shingles)


Shingles created for all documents.
{'doc3.txt': {'machine', 'is', 'area', 'a', 'of', 'science', 'key', 'learning', 'data'}, 'doc1.txt': {'an', 'is', 'science', 'interesting', 'topic', 'data'}, 'doc2.txt': {'an', 'is', 'science', 'of', 'fascinating', 'interesting', 'topic', 'very', 'the', 'subject', 'data'}}


In [ ]:
#MinHash Signatures
def minhash_signature(shingle_set,num_hashes=20):
    max_hash=1000
    signatures=[]
    random.seed(42)
    hash_funcs=[(random.randint(1,max_hash),random.randint(0,max_hash))for _ in range(num_hashes)]
    for a,b in hash_funcs:
        min_hash=max_hash
        for sh in shingle_set:
            h=int(hashlib.md5(sh.encode()).hexdigest(),16)
            val=(a*h+b)%max_hash
            if val<min_hash:
                min_hash=val
        signatures.append(min_hash)
    return signatures
signatures={doc: minhash_signature(s) for doc,s in shingles.items()}
print("\nMinHash signatures created.")
print(signatures)


MinHash signatures created.
{'doc3.txt': [14, 143, 86, 162, 4, 48, 134, 14, 4, 188, 302, 14, 82, 121, 152, 20, 137, 80, 32, 163], 'doc1.txt': [34, 143, 20, 178, 99, 115, 134, 14, 4, 130, 174, 170, 154, 345, 152, 20, 69, 80, 326, 25], 'doc2.txt': [14, 23, 20, 162, 4, 98, 122, 14, 4, 130, 174, 170, 154, 81, 152, 20, 21, 80, 326, 25]}


In [ ]:
#LSH Candidate Pairs
def lsh(signatures,bands):
    num_hashes=len(next(iter(signatures.values())))
    rows=num_hashes//bands
    buckets={}
    candidate_pairs=set()
    for doc,sig in signatures.items():
        for b in range(bands):
            start=b*rows
            end=start+rows
            band=sig[start:end]
            bucket_ids=[h%50 for h in band]
            for bucket in bucket_ids:
                if bucket not in buckets:
                    buckets[bucket]=[]
                for other_doc in buckets[bucket]:
                    if other_doc!=doc:
                        candidate_pairs.add(tuple(sorted([doc,other_doc])))
                buckets[bucket].append(doc)
    return candidate_pairs
candidates=lsh(signatures,bands=5)
print("\nLSH Candidate Pairs:",candidates)


LSH Candidate Pairs: {('doc1.txt', 'doc2.txt'), ('doc2.txt', 'doc3.txt'), ('doc1.txt', 'doc3.txt')}


In [ ]:
#MinHash Similarity
def minhash_similarity(sig1,sig2):
    matches=sum(1 for i in range(len(sig1)) if sig1[i]==sig2[i])
    return matches/len(sig1)

In [ ]:
#Compute similarity only for candidate pairs
results=[]
for doc1,doc2 in candidates:
    sim=minhash_similarity(signatures[doc1],signatures[doc2])
    status="Plagiarized" if sim>=0.6 else "Not Plagiarized"
    results.append((doc1,doc2,round(sim,3),status))
df=pd.DataFrame(results,columns=["Document 1","Document 2","MinHash Similarity","Result"])
df.sort_values(by="MinHash Similarity", ascending=False, inplace=True)
print("\nPlagiarism Results:\n")
print(df.to_string(index=False))


Plagiarism Results:

Document 1 Document 2  MinHash Similarity          Result
  doc1.txt   doc2.txt                0.60     Plagiarized
  doc2.txt   doc3.txt                0.40 Not Plagiarized
  doc1.txt   doc3.txt                0.35 Not Plagiarized
